# spuriosity quickstart

This notebook walks through a 4-step end-to-end stress test:

1. **Build a panel** with a known data-generating process (DGP) and an injected confounder
2. **Run a `StressTest`** with OLS as the reference fit
3. **Plot coefficient recovery** vs. ground truth
4. **Compare multiple models** on the same DGP

Estimated runtime: ~10 seconds on a laptop.


In [ ]:
import spuriosity
from spuriosity import PanelGenerator, StressTest, compare_models, reference, plot_recovery_report

print(f"spuriosity v{spuriosity.__version__}")

# Build a 5,000-row panel with a known DGP and one injected pathology (a confounder).
gen = PanelGenerator(n_entities=500, n_periods=10, seed=42)
gen.add_variable("x1", dist="normal", mean=0, std=1)
gen.add_treatment("treat", assignment="random", start_period=0)
gen.set_outcome(
    formula="x1 + treat",
    coefficients={"x1": 2.0, "treat": 3.0, "Intercept": 1.0},
    noise_std=1.0,
)
gen.add_confounder(feature="x1", outcome="y", strength=0.6, observed=True)

print(gen)  # PanelGenerator.__repr__ shows the full builder state


In [ ]:
df, truth = gen.generate()

print(f"Generated panel: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print()
print(truth)  # __repr__ of the ground truth
df.head()


In [ ]:
test = StressTest(truth)
report = test.evaluate(
    fit_fn=reference.ols_fit,
    predict_fn=reference.ols_predict,
    data=df,
    fit_kwargs={"formula": "y ~ x1 + treat"},  # OLS needs full patsy (LHS + RHS)
    model_name="naive_OLS",
)
report.summary()


In [ ]:
# save_path= writes the PNG to disk; Figure is always returned for further use.
fig = plot_recovery_report(report, save_path="01_quickstart_recovery.png")
print(f"Figure saved to {fig.bbox_inches}")
fig


In [ ]:
# The confounder (strength=0.6) inflates naive_OLS's x1 coefficient.
# A 'controlled' OLS that includes the observed confounder recovers the truth.
cmp = compare_models(
    data=df, truth=truth,
    models={
        "naive_OLS":       (reference.ols_fit, reference.ols_predict),
        "controlled_OLS":  (reference.ols_fit, reference.ols_predict),
    },
    fit_kwargs_per_model={
        "naive_OLS":      {"formula": "y ~ x1 + treat"},
        "controlled_OLS": {"formula": "y ~ x1 + treat + _confounder_x1"},
    },
)
cmp.summary()


## Next steps

You just stress-tested OLS against a confounded DGP. The `controlled_OLS`
that knows about the confounder recovers the true coefficient; the naive
OLS doesn't. That's the loop: **define a DGP, fit a model, check if it
recovers the truth.** spuriosity is the toolkit for automating it.

Things to try:

- **Swap the pathology**: `add_structural_break(period=5, target="y", kind="mean_shift", magnitude=2.0)` and see how the fitted coefficients change before/after the break
- **Swap the reference fit**: `reference.sklearn_lr_fit(data, features=["x1", "treat"], target="y")` for a sklearn baseline
- **Heterogeneous treatment effects**: `add_hte(treatment="treat", modifier="x1", formula="3.0 + 1.5*x1")` to give the treatment effect a real CATE function
- **Read the docs**: [`docs/design_spec.md`](../docs/design_spec.md) for the full design rationale; the README's *Common gotchas* section for the 5 things that bite first-time users

**Found a bug or have a question?** [Open an issue](https://github.com/Nityahapani/spuriosity/issues/new) -- the maintainer is responsive.
